# Task 3: Combine Task 1 and Task 2 (stacked detector)

**Idea.** Task 1 looks at how confident the model was; Task 2 looks at whether its samples agree. They might catch different mistakes, so we train a small logistic regression on all four scores: `mean_logprob`, `mean_entropy` (Task 1) and `selfcheck_bert`, `selfcheck_nli` (Task 2).

* 80/20 split, stratified so both parts have the same share of hallucinations, seed 42.
* Features are standardized (they have very different scales), then logistic regression.
* AUC-PR on the held-out 20%, next to Task 1 alone and Task 2 alone on the same 20%.

The test set is small (about 164 questions), so one split can be lucky or unlucky. We also repeat the split 100 times to see how stable the comparison is.

Needs `outputs/task2/`. CPU is enough, about 1 minute. Outputs: `outputs/task3/`.

In [ ]:
# save everything to google drive so task 2 and task 3 can read it later
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/project7/outputs', exist_ok=True)
os.chdir('/content/drive/MyDrive/project7')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, precision_recall_curve

SEED = 42
OUT = "outputs/task3"
os.makedirs(OUT, exist_ok=True)
T1 = ["mean_logprob", "mean_entropy"]
T2 = ["selfcheck_bert", "selfcheck_nli"]

d = pd.read_csv("outputs/task2/task2_features.csv").dropna(subset=T1 + T2).reset_index(drop=True)
print(len(d), "questions, hallucinated share:", round(d.hallucinated.mean(), 3))

In [ ]:
train, test = train_test_split(d, test_size=0.2, random_state=SEED, stratify=d["hallucinated"])
pd.DataFrame({"id": d["id"], "part": np.where(d.index.isin(test.index), "test", "train")}).to_csv(f"{OUT}/split.csv", index=False)

def model():
    return make_pipeline(StandardScaler(), LogisticRegression())

def test_scores(train, test):
    # every detector's hallucination score on the test part (higher = more likely hallucinated)
    s = {"Task 1: -mean log-prob": -test["mean_logprob"],
         "Task 1: mean entropy": test["mean_entropy"],
         "Task 2: SelfCheck-BERTScore": test["selfcheck_bert"],
         "Task 2: SelfCheck-NLI": test["selfcheck_nli"]}
    for name, cols in [("LR on Task 1 features", T1), ("LR on Task 2 features", T2), ("Task 3: LR on all four", T1 + T2)]:
        s[name] = model().fit(train[cols], train["hallucinated"]).predict_proba(test[cols])[:, 1]
    return s

scores = test_scores(train, test)
yt = test["hallucinated"].values
results = pd.DataFrame({"AUC-PR (test 20%)": {k: average_precision_score(yt, v) for k, v in scores.items()}})
results.loc["random"] = yt.mean()

## How stable is this? Repeat the split 100 times

For each split we compare Task 3 with SelfCheck-NLI on the *same* test questions (a paired comparison). The mean and std of that difference say whether combining really helps.

In [ ]:
runs = []
for k in range(100):
    tr, te = train_test_split(d, test_size=0.2, random_state=k, stratify=d["hallucinated"])
    runs.append({n: average_precision_score(te["hallucinated"], v) for n, v in test_scores(tr, te).items()})
runs = pd.DataFrame(runs)
results["mean of 100 splits"] = runs.mean()
results["std"] = runs.std()
results.loc["random", "mean of 100 splits"] = d.hallucinated.mean()

diff = runs["Task 3: LR on all four"] - runs["Task 2: SelfCheck-NLI"]
print(f"Task 3 beats SelfCheck-NLI alone in {(diff > 0).mean():.0%} of the 100 splits")
print(f"paired difference Task 3 - NLI: mean {diff.mean():+.3f}, std {diff.std():.3f}")
results.to_csv(f"{OUT}/task3_results.csv")
results.round(3)

## What did the model learn?

`mean_logprob` and `mean_entropy` are almost the same signal (correlation about -0.97). When two inputs are that correlated, logistic regression can split the weight between them in many ways, so their individual coefficients (and even their signs) can't be read as importance. To read the weights, we refit with only one of the two (mean entropy) next to the Task 2 features. The AUC-PR barely changes, so nothing is lost.

In [ ]:
print("correlation between the features:")
print(d[T1 + T2].corr().round(2))

stack = model().fit(train[T1 + T2], train["hallucinated"])
print("\ncoefficients, all four (not interpretable one by one):")
print(pd.Series(stack[-1].coef_[0], index=T1 + T2).round(3))

cols = ["mean_entropy"] + T2
small = model().fit(train[cols], train["hallucinated"])
print("\ncoefficients without mean_logprob:")
print(pd.Series(small[-1].coef_[0], index=cols).round(3))
print("test AUC-PR without mean_logprob:", round(average_precision_score(yt, small.predict_proba(test[cols])[:, 1]), 3))

In [ ]:
for name in ["Task 1: mean entropy", "Task 2: SelfCheck-NLI", "Task 3: LR on all four"]:
    p, r, _ = precision_recall_curve(yt, scores[name])
    plt.plot(r, p, label=f"{name} ({average_precision_score(yt, scores[name]):.3f})")
plt.axhline(yt.mean(), ls="--", c="gray", label=f"random ({yt.mean():.3f})")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("Held-out 20%: Task 1 vs Task 2 vs Task 3"); plt.legend(fontsize=8)
plt.savefig(f"{OUT}/task3_pr.png", dpi=150); plt.show()

## Examples where combining helped or hurt (for the report)

Rank of each test question from 0 (least suspicious) to 1 (most suspicious), for NLI alone and for the stack.

In [ ]:
pd.set_option("display.max_colwidth", 120)
t = test.copy()
t["rank_nli"] = pd.Series(scores["Task 2: SelfCheck-NLI"].values, index=t.index).rank(pct=True)
t["rank_stack"] = pd.Series(scores["Task 3: LR on all four"], index=t.index).rank(pct=True)
t["moved"] = t["rank_stack"] - t["rank_nli"]
cols = ["question", "answer", "mean_entropy", "selfcheck_nli", "rank_nli", "rank_stack"]
print("hallucinations the stack ranked higher than NLI did:")
print(t[t.hallucinated == 1].nlargest(3, "moved")[cols].round(2).to_string())
print("\nhallucinations the stack ranked lower than NLI did:")
print(t[t.hallucinated == 1].nsmallest(3, "moved")[cols].round(2).to_string())